### Data Preparation

In the previous phase (Data Understanding), I explored the Titanic dataset and identified several data quality issues that may affect model performance and interpretation.

The key issues discovered include:

- Column spelling inconsistencies (e.g., `2urvived`, `sibsp`)
- Presence of unnecessary columns starting with `"zero"`
- Outliers in the `Fare` variable
- Skewness in numeric variables such as `Fare`
- Categorical variables that require encoding (e.g., `Sex`, `Embarked`)
- Need for feature engineering (e.g., Family Size, Fare Bands)

To address these issues and prepare the dataset for modeling, I will follow a structured data preparation approach aligned with the CRISP-DM methodology.

The following steps will be applied:

1. Select Data – Keep only relevant and useful columns.
2. Clean Data – Fix spelling issues, remove unnecessary columns, and handle inconsistencies.
3. Construct Data – Create new meaningful features such as Family Size and Fare brackets.
4. Integrate Data – Merge additional datasets if necessary.
5. Format Data – Ensure correct data types and encode categorical variables for modeling.

This structured process ensures that the dataset is clean, consistent, and ready for machine learning modeling. I'll start by loading the raw dataset.


In [1]:
# Import libraries
import pandas as pd
import numpy as np

from titanic_surv.dataset import load_data

# Load the Titanic dataset
data = load_data("titanic.csv")

# Display basic statistics for numeric columns
data.head()

2026-02-12 02:23:45.562 | INFO     | titanic_surv.config:<module>:11 - PROJ_ROOT path is: D:\py\Titanic\titanic_surv_pred_insights


,Passengerid,Age,Fare,Sex,sibsp,zero,zero.1,zero.2,zero.3,zero.4,...,zero.12,zero.13,zero.14,Pclass,zero.15,zero.16,Embarked,zero.17,zero.18,2urvived
0,1,22.0,7.2500,0,1,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0
1,2,38.0,71.2833,1,1,0,0,0,0,0,...,0,0,0,1,0,0,0.0,0,0,1
2,3,26.0,7.9250,1,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,1
3,4,35.0,53.1000,1,1,0,0,0,0,0,...,0,0,0,1,0,0,2.0,0,0,1
4,5,35.0,8.0500,0,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0


#### 1. Select Data

The purpose of the *Select Data* step is to choose only the columns that are relevant to the project objective.

From the Data Understanding phase, I identified that not all columns in the raw dataset are useful for predicting survival or generating insights. Some columns:
- Start with `"zero"` and do not provide meaningful information
- Are not relevant to survival prediction

To keep the analysis focused and reduce noise, i select only the key passenger attributes that are known to influence survival outcomes.

In [3]:
# Select relevant columns for analysis
selected_columns = [
    "Passengerid",
    "Age",
    "Fare",
    "Sex",
    "sibsp",
    "Parch",
    "Pclass",
    "Embarked",
    "2urvived"
]

# Create a working copy of the dataset
data_selected = data[selected_columns].copy()

# Check selected columns
data_selected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Passengerid  1309 non-null   int64  
 1   Age          1309 non-null   float64
 2   Fare         1309 non-null   float64
 3   Sex          1309 non-null   int64  
 4   sibsp        1309 non-null   int64  
 5   Parch        1309 non-null   int64  
 6   Pclass       1309 non-null   int64  
 7   Embarked     1307 non-null   float64
 8   2urvived     1309 non-null   int64  
dtypes: float64(3), int64(6)
memory usage: 92.2 KB


#### 2. Clean Data

The goal of the *Clean Data* step is to fix errors, remove unnecessary information, and ensure the dataset is consistent and reliable for analysis and modeling.
From the Data Quality verification step, the following issues were identified:

- Column spelling errors (e.g., `2urvived`, `sibsp`)
- Unnecessary columns starting with `"zero"`
- Inconsistent column name formatting
- Potential data type inconsistencies

In this step, I will:
1. Normalize column names
2. Fix spelling errors in column names
3. Verify data types after cleaning
4. Fill in missing categorical values with mode if any
5. Fill missing numeric values with median if any

##### Step 1: Normalize Column Names

In [4]:
# Standardize column names: remove spaces and ensure consistency
data_clean = data_selected.copy()
data_clean.columns = data_clean.columns.str.strip()

##### Step 2: Fix Column Spelling Errors and Format

In [5]:
# Rename incorrect column names to standard Titanic naming
data_clean = data_clean.rename(columns={
    "Passengerid": "PassengerId",
    "2urvived": "Survived",
    "sibsp": "SibSp"
})

# Verify column names
data_clean.columns

Index(['PassengerId', 'Age', 'Fare', 'Sex', 'SibSp', 'Parch', 'Pclass',
       'Embarked', 'Survived'],
      dtype='object')

##### Step 3: Verify and Fix Data Types

In [6]:
# Ensure correct data types
data_clean["Survived"] = data_clean["Survived"].astype(int)
data_clean["Pclass"] = data_clean["Pclass"].astype(int)
data_clean["SibSp"] = data_clean["SibSp"].astype(int)
data_clean["Parch"] = data_clean["Parch"].astype(int)
data_clean["Fare"] = data_clean["Fare"].astype(float)

# Check current data types
data_clean.dtypes

PassengerId      int64
Age            float64
Fare           float64
Sex              int64
SibSp            int64
Parch            int64
Pclass           int64
Embarked       float64
Survived         int64
dtype: object

##### Step 4: Fill in missing categorical values

In [7]:
data_clean["Sex"] = data_clean["Sex"].fillna(data_clean["Sex"].mode()[0])
data_clean["Embarked"] = data_clean["Embarked"].fillna(data_clean["Embarked"].mode()[0])

##### Step 5: Fill missing numeric values with median

In [8]:
data_clean["Age"] = data_clean["Age"].fillna(data_clean["Age"].median())
data_clean["Fare"] = data_clean["Fare"].fillna(data_clean["Fare"].median())

##### Step 6: Final Clean Data Check

In [9]:
# Summary after cleaning
data_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Age          1309 non-null   float64
 2   Fare         1309 non-null   float64
 3   Sex          1309 non-null   int64  
 4   SibSp        1309 non-null   int64  
 5   Parch        1309 non-null   int64  
 6   Pclass       1309 non-null   int64  
 7   Embarked     1309 non-null   float64
 8   Survived     1309 non-null   int64  
dtypes: float64(3), int64(6)
memory usage: 92.2 KB


#### 3. Construct Data

The purpose of this step is to create new features from existing data to improve understanding and modeling performance.

Based on our analysis, I will construct the following new features:

1. **Family Size** – Total number of family members aboard
2. **Is Alone** – Indicates whether a passenger traveled alone
3. **Fare Bracket** – Groups fares into meaningful categories
4. **Passenger Class Label** – Human-readable class names
5. **Age Group** – Categorizes passengers into life-stage groups

These new features help simplify complex patterns and make the dataset more meaningful for analysis and modeling.

##### Step 1: Construct Family Size

In [10]:
# Family Size = siblings/spouses + parents/children + passenger
data_clean["Family_Size"] = data_clean["SibSp"] + data_clean["Parch"] + 1

# Preview result
data_clean[["SibSp", "Parch", "Family_Size"]].head()


,SibSp,Parch,Family_Size
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


##### Step 2: Construct "Is Alone" Feature

In [11]:
# Create a binary feature: 1 if passenger is alone, else 0
data_clean["Is_Alone"] = (data_clean["Family_Size"] == 1).astype(int)

data_clean[["Family_Size", "Is_Alone"]].head()

,Family_Size,Is_Alone
0,2,0
1,2,0
2,1,1
3,2,0
4,1,1


##### Step 3: Construct Fare Bracket

In [13]:
# Create fare bands using quartiles
data_clean["Fare_Bracket"] = pd.qcut(
    data_clean["Fare"],
    q=4,
    labels=["Low Fare", "Medium Fare", "High Fare", "Very High Fare"]
)

# Check distribution
data_clean["Fare_Bracket"].value_counts()

Fare_Bracket
Low Fare          337
High Fare         328
Very High Fare    323
Medium Fare       321
Name: count, dtype: int64

##### Step 4: Create Passenger Class Labels

In [14]:
# Map passenger class to readable labels
data_clean["Pclass_Label"] = data_clean["Pclass"].map({
    1: "First Class",
    2: "Second Class",
    3: "Third Class"
})

data_clean[["Pclass", "Pclass_Label"]].head()

,Pclass,Pclass_Label
0,3,Third Class
1,1,First Class
2,3,Third Class
3,1,First Class
4,3,Third Class


##### Step 5: Create Age Group

In [15]:
# Create Age Groups
age_bins = [0, 12, 18, 35, 60, 80]
age_labels = ["Child", "Teen", "Young Adult", "Adult", "Senior"]

data_clean["Age_Group"] = pd.cut(
    data_clean["Age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

# Check distribution
data_clean["Age_Group"].value_counts()

Age_Group
Young Adult    794
Adult          289
Teen            99
Child           94
Senior          33
Name: count, dtype: int64

##### Step 6: Final Check of Constructed Features

In [16]:
# Verify new columns
data_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   PassengerId   1309 non-null   int64   
 1   Age           1309 non-null   float64 
 2   Fare          1309 non-null   float64 
 3   Sex           1309 non-null   int64   
 4   SibSp         1309 non-null   int64   
 5   Parch         1309 non-null   int64   
 6   Pclass        1309 non-null   int64   
 7   Embarked      1309 non-null   float64 
 8   Survived      1309 non-null   int64   
 9   Family_Size   1309 non-null   int64   
 10  Is_Alone      1309 non-null   int64   
 11  Fare_Bracket  1309 non-null   category
 12  Pclass_Label  1309 non-null   object  
 13  Age_Group     1309 non-null   category
dtypes: category(2), float64(3), int64(8), object(1)
memory usage: 125.8+ KB


#### 4. Integrate Data

Data integration involves combining data from multiple sources to create a unified dataset for analysis or modeling.

In this project, the Titanic dataset already contains all the required variables needed to predict passenger survival, such as passenger class, age, family relationships, and fare information.

During the data understanding phase, I considered the possibility of enriching the dataset using external sources (e.g., Kaggle Titanic datasets). However, after reviewing the current dataset, I observed that:

- All essential features for modeling are already present.
- No additional external data is required to improve prediction quality.
- Feature engineering has already enhanced the dataset by creating meaningful variables such as Family Size, Is Alone, Age Group, and Fare Bracket.

Therefore, **no external data to integrat**


##### Confirm dataset shape after preparation

In [18]:
data_clean.shape

(1309, 14)

##### Preview final dataset columns

In [19]:
data_clean.columns.tolist()

['PassengerId',
 'Age',
 'Fare',
 'Sex',
 'SibSp',
 'Parch',
 'Pclass',
 'Embarked',
 'Survived',
 'Family_Size',
 'Is_Alone',
 'Fare_Bracket',
 'Pclass_Label',
 'Age_Group']

##### Check if PassengerId is Unique

In [20]:
# Check for duplicate PassengerId after preparation
data_clean["PassengerId"].is_unique

True

#### 5. Format Data

The purpose of formatting data is to prepare it for machine learning modeling. This ensures that all features are in the correct type and structure, and that categorical variables are converted to a format models can understand.

In this project, the formatting steps include:

1. Correct Data Types
2. Encode Categorical Variables
3. Ensure Target Variable is Correct  
4. Check final dataset info


##### Copy dataset to avoid modifying original

In [21]:
formatted_data_clean = data_clean.copy()
formatted_data_clean.head()

,PassengerId,Age,Fare,Sex,SibSp,Parch,Pclass,Embarked,Survived,Family_Size,Is_Alone,Fare_Bracket,Pclass_Label,Age_Group
0,1,22.0,7.2500,0,1,0,3,2.0,0,2,0,Low Fare,Third Class,Young Adult
1,2,38.0,71.2833,1,1,0,1,0.0,1,2,0,Very High Fare,First Class,Adult
2,3,26.0,7.9250,1,0,0,3,2.0,1,1,1,Medium Fare,Third Class,Young Adult
3,4,35.0,53.1000,1,1,0,1,2.0,1,2,0,Very High Fare,First Class,Young Adult
4,5,35.0,8.0500,0,0,0,3,2.0,0,1,1,Medium Fare,Third Class,Young Adult


##### Correct Data Types
- Ensure numeric columns are of type `int` or `float`.
- Ensure categorical columns are `category` or encoded properly.

In [22]:
numeric_cols = ["Age", "SibSp", "Parch", "Fare", "Family_Size"]
for col in numeric_cols:
    formatted_data_clean[col] = pd.to_numeric(formatted_data_clean[col], errors='coerce')

categorical_cols = ["Survived", "Pclass", "Sex", "Embarked", "Is_Alone", "Age_Group", "Fare_Bracket"]
for col in categorical_cols:
    formatted_data_clean[col] = formatted_data_clean[col].astype("category")

categorical_cols

['Survived',
 'Pclass',
 'Sex',
 'Embarked',
 'Is_Alone',
 'Age_Group',
 'Fare_Bracket']

##### Encode Categorical Variables
- Replace an actual category for label - Sex (male"0"/female"1")
- Convert text labels (like "Sex" or "Pclass_Label") to numeric codes if needed.

In [23]:
# Format Sex column
formatted_data_clean["Sex"] = formatted_data_clean["Sex"].astype(str).str.strip().str.lower()
formatted_data_clean["Sex"] = formatted_data_clean["Sex"].replace({"0": "male", "1": "female"})  # map existing 0/1 to text
formatted_data_clean["Sex"] = formatted_data_clean["Sex"].where(
     formatted_data_clean["Sex"].isin(["male", "female"]), "male"
)

formatted_data_clean["Sex_Code"] = formatted_data_clean["Sex"].map({"male": 0, "female": 1}).astype(int)

# Ensure Pclass is numeric
formatted_data_clean["Pclass"] = pd.to_numeric(formatted_data_clean["Pclass"], errors="coerce")
pclass_mode = formatted_data_clean["Pclass"].mode()[0]
formatted_data_clean["Pclass"] = formatted_data_clean["Pclass"].fillna(pclass_mode)
formatted_data_clean["Pclass_Code"] = formatted_data_clean["Pclass"].astype(int)

formatted_data_clean.head()

,PassengerId,Age,Fare,Sex,SibSp,Parch,Pclass,Embarked,Survived,Family_Size,Is_Alone,Fare_Bracket,Pclass_Label,Age_Group,Sex_Code,Pclass_Code
0,1,22.0,7.2500,male,1,0,3,2.0,0,2,0,Low Fare,Third Class,Young Adult,0,3
1,2,38.0,71.2833,female,1,0,1,0.0,1,2,0,Very High Fare,First Class,Adult,1,1
2,3,26.0,7.9250,female,0,0,3,2.0,1,1,1,Medium Fare,Third Class,Young Adult,1,3
3,4,35.0,53.1000,female,1,0,1,2.0,1,2,0,Very High Fare,First Class,Young Adult,1,1
4,5,35.0,8.0500,male,0,0,3,2.0,0,1,1,Medium Fare,Third Class,Young Adult,0,3


##### Ensure Target Variable is Correct

In [24]:
formatted_data_clean["Survived"] = formatted_data_clean["Survived"].astype("int64")

##### Check final dataset info

In [25]:
# Quick look at a few rows to confirm encoding
formatted_data_clean.head()

,PassengerId,Age,Fare,Sex,SibSp,Parch,Pclass,Embarked,Survived,Family_Size,Is_Alone,Fare_Bracket,Pclass_Label,Age_Group,Sex_Code,Pclass_Code
0,1,22.0,7.2500,male,1,0,3,2.0,0,2,0,Low Fare,Third Class,Young Adult,0,3
1,2,38.0,71.2833,female,1,0,1,0.0,1,2,0,Very High Fare,First Class,Adult,1,1
2,3,26.0,7.9250,female,0,0,3,2.0,1,1,1,Medium Fare,Third Class,Young Adult,1,3
3,4,35.0,53.1000,female,1,0,1,2.0,1,2,0,Very High Fare,First Class,Young Adult,1,1
4,5,35.0,8.0500,male,0,0,3,2.0,0,1,1,Medium Fare,Third Class,Young Adult,0,3


#### 6. Split Dataset

Now that the data is clean, formatted, and all features are ready, I need to split it into **training, validation, and test sets**.  

- **Training Set (50%)**: Used to train the machine learning model.  
- **Validation Set (30%)**: Used to check the model's performance during training and tune parameters.  
- **Test Set (20%)**: Used only at the end to evaluate the model on completely unseen data.  

I will save each dataset as a separate CSV file for later use.


In [27]:
from sklearn.model_selection import train_test_split
from titanic_surv.dataset import save_processed_data

# Define features and target
X = formatted_data_clean.drop(columns=["Survived"])
y = formatted_data_clean["Survived"]

##### Step 1: Split into training (50%) and temp (50% for validation + test)

In [28]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.5, random_state=42, stratify=y
)

##### Step 2: Split temp into validation (30%) and test (20% of original)

In [29]:
# Note: validation = 30% / 50% temp = 0.6 of temp
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.4, random_state=42, stratify=y_temp
)

# Combine features and target back for saving
train_df = X_train.copy()
train_df["Survived"] = y_train

validate_df = X_val.copy()
validate_df["Survived"] = y_val

test_df = X_test.copy()
test_df["Survived"] = y_test

##### Step 3: Save the processed dataset

In [30]:
# Save processed datasets
save_processed_data(train_df, filename="titanic_training.csv")
save_processed_data(validate_df, filename="titanic_validate.csv")
save_processed_data(test_df, filename="titanic_test.csv")

# Print dataset sizes
print(f"Training set size: {train_df.shape[0]} samples")
print(f"Validation set size: {validate_df.shape[0]} samples")
print(f"Test set size: {test_df.shape[0]} samples")

2026-02-12 10:48:53.541 | SUCCESS  | titanic_surv.dataset:save_processed_data:33 - Processed data saved to D:\py\Titanic\titanic_surv_pred_insights\data\processed\titanic_training.csv
2026-02-12 10:48:53.545 | SUCCESS  | titanic_surv.dataset:save_processed_data:33 - Processed data saved to D:\py\Titanic\titanic_surv_pred_insights\data\processed\titanic_validate.csv
2026-02-12 10:48:53.549 | SUCCESS  | titanic_surv.dataset:save_processed_data:33 - Processed data saved to D:\py\Titanic\titanic_surv_pred_insights\data\processed\titanic_test.csv
Training set size: 654 samples
Validation set size: 393 samples
Test set size: 262 samples


#### 7. Pre-Modeling Validation Checklist

Before proceeding to modeling, I validate that:

1. The target variable is correct
2. Survived is binary (0 and 1 only)
3. All features are numeric and model-ready
4. Categorical variables have been encoded
5. No data leakage exists
6. Target is not included in feature matrix
7. Train, validation, and test sets are clearly separated
8. No row overlap exists between datasets

##### Confirm Target Variable Exists

In [33]:
assert "Survived" in formatted_data_clean.columns, "Target variable 'Survived' not found."

print("Target variable exists.")


Target variable exists.


##### Confirm Target Is Binary (0 and 1 Only)

In [53]:
unique_values = sorted(formatted_data_clean["Survived"].unique())

print("Unique target values:", unique_values)

assert set(unique_values) == {0, 1}, "Target variable is not binary (0 and 1 only)."

print("Target variable is correctly binary.")

Unique target values: [np.int64(0), np.int64(1)]
Target variable is correctly binary.


##### Confirm All Key Features Are Numeric

In [55]:
# feature_df = formatted_data_clean.drop(columns=["Survived"])
model_features = [
    "Age",
    "Fare",
    "SibSp",
    "Parch",
    "Family_Size",
    "Sex_Code",
    "Pclass_Code"
]

feature_df = formatted_data_clean[model_features]

# Check data types of model features
non_numeric_model_features = (
    feature_df
    .select_dtypes(exclude=["number"])
    .columns
    .tolist()
)

if len(non_numeric_model_features) == 0:
    print("All model features are numeric and model-ready.")
else:
    print("Non-numeric model features found:", non_numeric_model_features)

# Non-numeric columns found: ['Sex', 'Embarked', 'Is_Alone', 'Fare_Bracket', 'Pclass_Label', 'Age_Group']

All model features are numeric and model-ready.


##### Confirm Categorical Variables Are Encoded
If the previous step shows no non-numeric columns, then categorical encoding is confirmed.

In [56]:
print(feature_df.dtypes)

Age            float64
Fare           float64
SibSp            int64
Parch            int64
Family_Size      int64
Sex_Code         int64
Pclass_Code      int64
dtype: object


##### Check for Data Leakage

Data leakage often happens when:
- Target-related information is included in features
- Post-event variables are used

In [57]:
leakage_cols = [col for col in feature_df.columns if "surv" in col.lower()]

if len(leakage_cols) == 0:
    print("No obvious leakage columns detected.")
else:
    print("Possible leakage columns:", leakage_cols)


No obvious leakage columns detected.


##### Ensure Target Is Not Included in Feature Matrix

In [58]:
assert "Survived" not in feature_df.columns, "Target variable included in feature set!"

print("Target variable not included in features.")


Target variable not included in features.


##### Confirm Dataset Sizes (50 / 30 / 20)

In [60]:
total_rows = (
    len(train_df) + 
    len(validate_df) + 
    len(test_df)
)

original_rows = len(formatted_data_clean)

print("Original rows:", original_rows)
print("Total split rows:", total_rows)

assert total_rows == original_rows, "Split datasets do not sum to original dataset size."

print("Dataset split sizes are consistent.")


Original rows: 1309
Total split rows: 1309
Dataset split sizes are consistent.


##### Confirm No Overlap Between Datasets

In [61]:
train_index = set(train_df.index)
validate_index = set(validate_df.index)
test_index = set(test_df.index)

assert train_index.isdisjoint(validate_index), "Overlap between train and validation!"
assert train_index.isdisjoint(test_index), "Overlap between train and test!"
assert validate_index.isdisjoint(test_index), "Overlap between validation and test!"

print("No overlap between train, validation, and test datasets.")


No overlap between train, validation, and test datasets.
